In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib
import os

In [3]:
# Load data
filepath = r"C:\Users\vasth\Downloads\gwl_tel_6_hourly_meghalaya_ml_2021_2025.csv"
time_col = 'Data Acquisition Time'
target_col = 'Groundwater Level Telemetry 6 Hourly (meter)'

print("Loading data...")
df = pd.read_csv(filepath)

# Parse time, drop bad rows
df[time_col] = pd.to_datetime(df[time_col], format='%d-%m-%Y %H:%M', errors='coerce')
df = df.dropna(subset=[time_col, target_col])

# Remove sensor error codes (e.g. telemetry reporting 999 on fault)
df = df[(df[target_col] >= -50) & (df[target_col] < 150)]

# Sort chronologically per station
df = df.sort_values(by=['Station', time_col])

# Temporal features
df['Month'] = df[time_col].dt.month
df['Hour'] = df[time_col].dt.hour
df['DayOfYear'] = df[time_col].dt.dayofyear

# Lag features (previous readings, per station)
df['GWL_lag1'] = df.groupby('Station')[target_col].shift(1)
df['GWL_lag4'] = df.groupby('Station')[target_col].shift(4)

# Rate of change and rolling mean (built only from past values — no leakage)
df['GWL_diff_1'] = df.groupby('Station')[target_col].diff(1)
df['GWL_roll_mean_24h'] = df.groupby('Station')['GWL_lag1'].transform(
    lambda x: x.rolling(window=4, min_periods=1).mean()
)

# Drop rows with NaNs from lagging/rolling
df = df.dropna(subset=['GWL_lag1', 'GWL_lag4', 'GWL_diff_1', 'GWL_roll_mean_24h'])

# Encode station as a category
df['Station_Code'] = df['Station'].astype('category').cat.codes

features = ['Month', 'Hour', 'DayOfYear', 'GWL_lag1', 'GWL_lag4',
            'GWL_diff_1', 'GWL_roll_mean_24h', 'Station_Code']

X = df[features]
y = df[target_col]

print(f"Data shape after preprocessing: {X.shape}")
print(f"Using features: {features}")

Loading data...
Data shape after preprocessing: (50889, 8)
Using features: ['Month', 'Hour', 'DayOfYear', 'GWL_lag1', 'GWL_lag4', 'GWL_diff_1', 'GWL_roll_mean_24h', 'Station_Code']


In [4]:
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Trained XGBoost model on {len(X_train)} samples...")
# Initialize and train XGBoost model
xgb_model = XGBRegressor(
    n_estimators=300, 
    learning_rate=0.05, 
    max_depth=6, 
    subsample=0.8,
    colsample_bytree=0.8,
    enable_categorical=True,
    random_state=42, 
    n_jobs=-1
)
xgb_model.fit(X_train, y_train)

Trained XGBoost model on 40711 samples...


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=300,
             n_jobs=-1, num_parallel_tree=None, ...)

In [5]:
predictions = xgb_model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))

print(f"Mean Absolute Error (MAE): {mae:.4f} meters")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f} meters")

Mean Absolute Error (MAE): 0.7534 meters
Root Mean Squared Error (RMSE): 3.7665 meters
